# P04 · Generator Split COCO `val2017` (`samples_val.json` & `samples_test.json`)

Notebook ini menghasilkan pembagian subset **COCO `val2017`** yang **deterministik** dan **terpisah di level gambar** (tidak ada `image_id` yang bocor antara Validasi dan Uji), lengkap dengan koordinat titik awal `p1` (`p1_x`, `p1_y`) yang kompatibel dengan notebook evaluasi Gibran.

- **Runtime**: CPU saja sudah cukup (< 1 menit, tidak perlu GPU).
- **Output**: `coco_splits_p04.zip` berisi `samples_val.json` (100 objek), `samples_test.json` (300 objek), dan `split_metadata.json` untuk disimpan ke folder `configs/` di repo tim.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pycocotools", "opencv-python"],
    check=True,
)

work = Path("/content/p04_splits")
work.mkdir(parents=True, exist_ok=True)

repo_dir = work / "P04-SAM2-PA_DL"
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/RadityaRamadhanEka/P04-SAM2-PA_DL.git", str(repo_dir)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)

ann_path = work / "instances_val2017.json"
if not ann_path.is_file():
    ann_zip = work / "annotations_trainval2017.zip"
    print("Mengunduh annotations_trainval2017.zip...")
    urlretrieve(
        "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        ann_zip,
    )
    with ZipFile(ann_zip) as zf:
        with zf.open("annotations/instances_val2017.json") as src, ann_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    ann_zip.unlink()
print(f"Anotasi siap: {ann_path}")

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

work = Path("/content/p04_splits")
repo_dir = work / "P04-SAM2-PA_DL"
out_dir = work / "configs"

subprocess.run(
    [
        sys.executable,
        str(repo_dir / "src" / "dataset_coco.py"),
        "--annotations", str(work / "instances_val2017.json"),
        "--out-dir", str(out_dir),
        "--n-val", "100",
        "--n-test", "300",
        "--min-area", "1024",
        "--max-per-image", "3",
        "--seed", "42",
    ],
    check=True,
)

val_samples = json.loads((out_dir / "samples_val.json").read_text(encoding="utf-8"))
print("Contoh 2 entri pertama samples_val.json:")
print(json.dumps(val_samples[:2], indent=2))

zip_path = work / "coco_splits_p04.zip"
with ZipFile(zip_path, "w") as zf:
    for fname in ("samples_val.json", "samples_test.json", "split_metadata.json"):
        zf.write(out_dir / fname, arcname=fname)

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print(f"Split tersimpan di {zip_path}")